<center>

## <h2><strong><font color="navy">GOJEK REVIEW INSIGHT NOTEBOOK</font></strong></h2>

</center>

## Introduction

This notebook is a companion to the main Python portfolio project, not the primary workflow. The main implementation lives in the Python pipeline, while this notebook is kept as a secondary option for walkthroughs, demos, or quick testing.

The dataset used in this analysis is [Gojek App Reviews in Bahasa Indonesia](https://www.kaggle.com/datasets/ucupsedaya/gojek-app-reviews-bahasa-indonesia/). It includes user reviews, rating scores, app versions, and review timestamps. Using this dataset, the analysis applies Natural Language Processing (NLP) techniques to classify sentiment into positive, negative, or neutral categories that can help evaluate and improve the Gojek app experience.

## Load Data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv')
df.head()

## EDA & Preprocessing

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sum(df['appVersion'].str.startswith("4.8"))

A total of 8,091 user reviews come from Gojek app versions that start with `4.8`.

In [ ]:
# keep only the required columns
df = df[df['appVersion'].str.startswith("4.8")]
df = df.loc[:, ['userName', 'content', 'score']]

df.head()

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

# tokenization
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

# remove duplicates
df = df.dropna(subset=['content']).drop_duplicates()

# stopwords
stop_words = stopwords.words('indonesian') + stopwords.words('english') + ["yg", "gak", "ngisi", "udah", "d", "sih", "nya", "srg", "utk", "byk", "gk", "ga", "aja", "tp", "udh"]
df['content'] = df['content'].apply(lambda x: [word.lower() for word in word_tokenize(x) if (word.isalpha() and word.lower() not in stop_words)])

# normalize text
df['content'] = df['content'].apply(lambda x: ' '.join(x))

df.head()

**Sastrawi** is used for Indonesian-language text preprocessing such as stemming or word normalization, while **VaderSentiment** is used for automated sentiment scoring, especially in social-media-like text.

In [ ]:
!pip install Sastrawi
!pip install VaderSentiment

In [ ]:
# stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
df['content'] = df['content'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

df.head(5)

In [ ]:
# labeling
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

additional_lexicon_id = {
    'kecewa': -0.4,
    'rugi': -1,
    'buruk': -0.6,
    'jelek': -0.6,
    'lelet': -0.7,
    'gagal': -0.5,
    'parah': -0.6,
    'mahal': -0.3,
    'tolong': -0.1,
    'hilang': -0.3,
    'gajelas': -0.3,
    'gj': -0.3,
    'promo': 0.6,
    'kadang': -0.1,
    'maling': -0.5,
    'ganggu': 0.3,
    'sedot': -0.5,
    'bagus': 0.5,
    'pulsa': 0,
    'potong': -1,
    'baik': 0.5,
    'kntl': -1,
    'ngelag': -0.8,
    'salah': -0.5,
    'bintang': 0,
    'benerin': -0.4,
    'lambat': -0.8,
    'siput': -0.4,
    'mati': -0.7,
    'minimal': -0.3,
    'susah': -0.6,
    'nagih': -0.6,
    'capek': -0.7,
    'kacau': -0.3,
    'tagih': -0.3,
    'mantap': 1,
    'puas': 0.9,
    'sampah': -0.5,
    'sulit': -0.6,
    'aneh': -0.4,
}

analyzer.lexicon.update(additional_lexicon_id)

df['sentiment'] = df['content'].apply(lambda x: 'positive' if analyzer.polarity_scores(x)['compound'] > 0 else ('negative' if analyzer.polarity_scores(x)['compound'] < 0 else 'neutral'))

df

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['content'])

### Sentiment Analysis

In [ ]:
from wordcloud import WordCloud
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter

#### WordCloud

In [ ]:
df_neutral = df[df['sentiment'] == 'neutral']
all_words_neutral = ' '.join([twts for twts in df_neutral['content']])
wordcloud_neutral = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_neutral)

plt.imshow(wordcloud_neutral, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud for Neutral Sentiment')
plt.show()

The neutral word cloud suggests that users often mention terms such as "gojek," "bantu," and "aplikasi," highlighting the app's practical utility. Additional terms such as "driver," "cepat," and "mudah" also appear, pointing to a generally functional but emotionally moderate service experience.

In [ ]:
df_positive = df[df['sentiment'] == 'positive']
all_words_positive = ' '.join([twts for twts in df_positive['content']])
wordcloud_positive = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_positive)

plt.imshow(wordcloud_positive, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud for Positive Sentiment')
plt.show()

The positive word cloud shows that users frequently use terms such as "bagus," "baik," "mantap," and "promo," indicating strong satisfaction with service quality, app performance, and promotions. Words such as "good" and "driver" also stand out, reflecting appreciation for helpful drivers and an overall pleasant customer experience.

In [ ]:
df_negative = df[df['sentiment'] == 'negative']
all_words_negative = ' '.join([twts for twts in df_negative['content']])
wordcloud_negative = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_negative)

plt.imshow(wordcloud_negative, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud for Negative Sentiment')
plt.show()

The negative word cloud shows frequent terms such as "driver," "susah," "kecewa," and "parah," reflecting dissatisfaction with service quality, especially around driver experience. Terms such as "cancel," "mahal," and "error" also appear, suggesting complaints about order cancellations, pricing, and technical issues in the app.

#### Target Distribution

In [ ]:
temp = df.groupby('sentiment').count()['content'].reset_index().sort_values(by='content', ascending=False)
temp.style.background_gradient(cmap='inferno_r')

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x='sentiment', data=df)

The bar chart shows that neutral reviews appear most often, followed by positive reviews, with negative reviews representing the smallest group.

In [ ]:
fig = go.Figure(go.Funnelarea(
    text=temp.sentiment,
    values=temp.content,
    title={"position": "top center", "text": "Funnel Chart of Target Distribution"}
    ))
fig.show()

The funnel chart reinforces this distribution: 49.3% neutral, 31.4% positive, and 19.3% negative. Most users therefore express a neutral view of the Gojek app.

The `palettable` library (specifically `Pastel1_7`) is used to make the visualizations more attractive and easier to distinguish.

In [ ]:
!pip install palettable
from palettable.colorbrewer.qualitative import Pastel1_7

In [ ]:
unique_neutral_words = df_neutral['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_neutral_words.columns = ['words', 'count']
top_20_words = unique_neutral_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot for Neutral Sentiment')
plt.show()

The neutral donut plot shows that users most often mention words such as "bantu," "aplikasi," and "gojek," suggesting that the app is generally viewed as useful even when sentiment is neutral. Terms such as "driver," "mudah," and "pesan" reinforce the theme of functional convenience and routine service interaction.

In [ ]:
unique_positive_words = df_positive['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_positive_words.columns = ['words', 'count']
top_20_words = unique_positive_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot for Positive Sentiment')
plt.show()

The positive donut plot highlights words such as "bagus," "mantap," and "baik," which point to customer satisfaction with the overall quality of the Gojek service. Frequently mentioned terms such as "promo," "top," and "good" suggest that users also value strong promotions and dependable service quality.

In [ ]:
unique_negative_words = df_negative['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_negative_words.columns = ['words', 'count']
top_20_words = unique_negative_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot for Negative Sentiment')
plt.show()

The negative donut plot is dominated by terms such as "gojek," "driver," and "kecewa," highlighting dissatisfaction with the service experience, especially around driver behavior and app performance. Terms such as "mahal," "susah," and "saldo" also suggest discomfort with pricing, technical issues, and account-related friction.

#### Train-Test Split

The dataset is split into 80% training data and 20% test data.

In [ ]:
# splitting
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['sentiment'], test_size=0.2, random_state=42)
X_train.shape, X_test.shape

#### Target Resampling

SMOTE (Synthetic Minority Over-sampling Technique) is applied to balance the number of observations in each sentiment class.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

plt.figure(figsize=(12, 6))
sns.countplot(x=y_train)
plt.title('Target Distribution for Modeling')
plt.show()

Positive, negative, and neutral sentiment classes become balanced before model training begins.

## Model

A Random Forest model is used with several candidate parameters such as the number of trees (`n_estimators`), tree depth (`max_depth`), the minimum samples required to split a node (`min_samples_split`), and the minimum samples required at each leaf (`min_samples_leaf`). These combinations are tested with `RandomizedSearchCV`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# initialize the parameter grid
rf_param_grid = {'n_estimators': [50, 100, 200],
                 'max_depth': [None, 10, 20, 30],
                 'min_samples_split': [2, 5, 10],
                 'min_samples_leaf': [1, 2, 4]}

### Random Forest

In [ ]:
rf_model = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, n_iter=10, cv=5, scoring='accuracy', random_state=42)
rf_model.fit(X_train, y_train)

The RandomizedSearchCV tuning process finds `min_samples_split=5` as part of the best-performing Random Forest configuration.

### Model Evaluation

In [ ]:
# print best param
print("\nBest Parameters for Random Forest:", rf_model.best_params_)

The best Random Forest configuration uses 100 trees, a minimum of 5 samples for node splitting, a minimum of 1 sample per leaf, and no maximum tree depth. This setup provides the best balance between model complexity and predictive accuracy.

In [ ]:
# evaluate the model
from sklearn.metrics import classification_report

y_pred_rf = rf_model.best_estimator_.predict(X_test)

print("\n\nClassification Report for Random Forest (Tuned):")
print(classification_report(y_test, y_pred_rf))

#### 1. Precision
- **Negative (0.82)**:  
  Of all reviews predicted as negative, 82% are truly negative toward the Gojek app. The remaining 18% are classification errors that should have been neutral or positive.

- **Neutral (0.96)**:  
  Reviews predicted as neutral have a very high precision of 96%, meaning most neutral reviews are identified correctly.

- **Positive (0.98)**:  
  This is the strongest-performing class, with 98% of reviews predicted as positive actually belonging to the positive class.

#### 2. Recall
- **Negative (0.92)**:  
  The model successfully detects 92% of truly negative reviews, showing a strong ability to capture negative sentiment from users.

- **Neutral (0.96)**:  
  The model identifies 96% of all neutral reviews, indicating reliable recognition of the neutral class.

- **Positive (0.90)**:  
  Of all truly positive reviews, 90% are correctly detected. This is slightly lower than the other classes but still represents very strong performance.

#### 3. F1-Score
- **Negative (0.87)**:  
  The negative-class F1-score of 0.87 shows a strong balance between precision and recall when classifying negative reviews.

- **Neutral (0.96)**:  
  The neutral F1-score is very high at 0.96, indicating consistent and accurate recognition with few errors.

- **Positive (0.94)**:  
  The positive class reaches an F1-score of 0.94, showing the model is also highly consistent for positive predictions.

#### 4. Accuracy (0.93)
The model reaches an overall accuracy of **93%**, meaning that about 93% of the 1,618 predicted reviews are classified correctly. This indicates strong end-to-end performance for Gojek app sentiment analysis.

#### 5. Macro Average (class-agnostic average)
- Precision: **0.92**
- Recall: **0.93**
- F1-score: **0.92**

These values show that the model performs strongly across all sentiment categories without being overly influenced by class size.

### 6. Weighted Average (support-weighted average)
- Precision: **0.94**
- Recall: **0.93**
- F1-score: **0.94**

These values indicate that the model still performs well overall even when weighted by the larger neutral class.